# 03 · Gold: facts
Builds the fact tables of the star schema. Run after `01_build_dimensions`.

| Table | Grain | Measures |
| --- | --- | --- |
| `fact_sales` | one row per sales order line (order + product) | Quantity, UnitPrice, SalesAmount, CostAmount, ProfitAmount |
| `fact_sales_target` | one row per salesperson per target month | TargetAmount |

Every fact key must exist in its dimension, and `fact_sales` must keep every silver sales row. Otherwise the notebook fails.

In [0]:
%run ../../config/pipeline_config

In [0]:
%run ../../utils/common_functions

In [0]:
def date_key(c):
    return F.date_format(c, "yyyyMMdd").cast("int")

silver_sales = spark.table(f"{SILVER_SCHEMA}.sales")
silver_targets = spark.table(f"{SILVER_SCHEMA}.targets")

fact_sales = silver_sales.select(
    "SalesOrderNumber",
    date_key("OrderDate").alias("OrderDateKey"),
    "OrderDate",
    "ProductKey", "ResellerKey", "EmployeeKey", "SalesTerritoryKey",
    "Quantity", "UnitPrice",
    F.col("Sales").alias("SalesAmount"),
    F.col("Cost").alias("CostAmount"),
    F.round(F.col("Sales") - F.col("Cost"), 2).alias("ProfitAmount"),
)

# Targets come keyed by EmployeeID; resolve to the EmployeeKey used by dim_salesperson
fact_sales_target = (silver_targets
    .join(spark.table(f"{GOLD_SCHEMA}.dim_salesperson").select("EmployeeID", "EmployeeKey"), "EmployeeID", "left")
    .select("EmployeeKey", "EmployeeID",
            date_key("TargetMonth").alias("TargetMonthKey"), "TargetMonth",
            F.col("Target").alias("TargetAmount")))

In [0]:
def gold(name):
    return spark.table(f"{GOLD_SCHEMA}.{name}")

dim_date_keys = gold("dim_date").select(F.col("DateKey"))
dq = DataQualityChecker("gold")

# Row counts must match silver exactly
dq.check("fact_sales", "row count equals silver.sales", abs(fact_sales.count() - silver_sales.count()))
dq.check("fact_sales_target", "row count equals silver.targets", abs(fact_sales_target.count() - silver_targets.count()))

# Every foreign key must resolve (critical: a missing key would silently drop rows from reports)
for dim, key in [("dim_product", "ProductKey"), ("dim_reseller", "ResellerKey"),
                 ("dim_sales_territory", "SalesTerritoryKey"), ("dim_salesperson", "EmployeeKey")]:
    dq.not_null("fact_sales", fact_sales, [key])
    dq.referential("fact_sales", fact_sales, dim, gold(dim), key, critical=True)
dq.referential("fact_sales", fact_sales, "dim_date", dim_date_keys.withColumnRenamed("DateKey", "OrderDateKey"), "OrderDateKey", critical=True)

dq.not_null("fact_sales_target", fact_sales_target, ["EmployeeKey", "TargetMonthKey"])
dq.referential("fact_sales_target", fact_sales_target, "dim_date", dim_date_keys.withColumnRenamed("DateKey", "TargetMonthKey"), "TargetMonthKey", critical=True)

display(dq.results_df())
dq.assert_no_critical_failures()

In [0]:
for name, df in {"fact_sales": fact_sales, "fact_sales_target": fact_sales_target}.items():
    write_delta_table(df.withColumn("gold_load_timestamp", F.current_timestamp()),
                      f"{GOLD_SCHEMA}.{name}", f"{GOLD_PATH}{name}")